In [3]:
# ============================================================
# CELL 1 (02_satellite_extraction): Setup and extraction points
# What it does:
#   - imports libraries, sets paths, creates data/satellite/
#   - loads station_meta.csv and adds Chapakot (the ungauged home site)
#     as an 8th point, flagged gauged=False
#   - builds a 3x3 block of sample points around each location, offset
#     by 0.05 degrees (the CHIRPS pixel size) in lat and lon, so every
#     location gets its own pixel plus its 8 neighbors without needing
#     to know the exact grid alignment
#   - saves data/satellite/extraction_points.csv (72 points)
#   - checks whether the Earth Engine Python package is installed
# ============================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)

ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"
SAT_DIR = ROOT / "data" / "satellite"
SAT_DIR.mkdir(parents=True, exist_ok=True)

meta = pd.read_csv(PROCESSED_DIR / "station_meta.csv", index_col="station")
pts = meta[["lat", "lon", "elev_m"]].copy()
pts["gauged"] = True

chap = pd.DataFrame(
    {"lat": [27.885457], "lon": [83.822247], "elev_m": [452], "gauged": [False]},
    index=pd.Index(["Chapakot"], name="station"),
)
pts = pd.concat([pts, chap])
print("Locations:")
print(pts.to_string())

# --- 3x3 block of sample points around each location ---
STEP = 0.05
rows = []
for st, r in pts.iterrows():
    for dj in (-1, 0, 1):          # north-south offset in pixels
        for di in (-1, 0, 1):      # east-west offset in pixels
            rows.append({
                "station": st,
                "pt_id": f"{st}|{di:+d}|{dj:+d}",
                "di": di, "dj": dj,
                "lat": round(r["lat"] + dj * STEP, 6),
                "lon": round(r["lon"] + di * STEP, 6),
                "is_center": (di == 0 and dj == 0),
            })
grid = pd.DataFrame(rows)
grid.to_csv(SAT_DIR / "extraction_points.csv", index=False)
print("\nExtraction points:", grid.shape, "-> saved data/satellite/extraction_points.csv")
print(grid.head(10).to_string(index=False))

# --- Earth Engine availability ---
try:
    import ee
    print("\nearthengine-api installed, version:", ee.__version__)
except ImportError:
    print("\nearthengine-api is NOT installed yet. In Anaconda Prompt, with the rainfall env active, run:")
    print("    conda install -c conda-forge earthengine-api")

Locations:
                       lat        lon  elev_m  gauged
station                                              
Pokhara Airport  28.188621  84.016925     804    True
Kunchha          28.126778  84.344942     820    True
Bandipur         27.941833  84.406378     991    True
Lumle            28.296539  83.817911    1738    True
Damauli          27.973675  84.265092     347    True
Sikles           28.355852  84.102068    1967    True
Lamachaur        28.261298  83.968287     991    True
Chapakot         27.885457  83.822247     452   False

Extraction points: (72, 7) -> saved data/satellite/extraction_points.csv
        station                 pt_id  di  dj       lat       lon  is_center
Pokhara Airport Pokhara Airport|-1|-1  -1  -1 28.138621 83.966925      False
Pokhara Airport Pokhara Airport|+0|-1   0  -1 28.138621 84.016925      False
Pokhara Airport Pokhara Airport|+1|-1   1  -1 28.138621 84.066925      False
Pokhara Airport Pokhara Airport|-1|+0  -1   0 28.188621 83.966925  

In [4]:
# ============================================================
# CELL 2 (02_satellite_extraction): Earth Engine login + small test extraction
# What it does:
#   1) initializes Earth Engine with your Cloud project ID (edit
#      EE_PROJECT); if that fails, opens the browser sign-in once and
#      retries
#   2) loads CHIRPS v3 daily "rnl" (ERA5-based) and prints: band name,
#      pixel size, number of daily images for 1983-2023 (expect 14975),
#      and the timestamp of one image
#   3) extracts the 72 sample points for 2021-07-05 .. 2021-07-09
#   4) prints the CHIRPS value at each station's centre pixel next to
#      the gauge value for the same dates (rain_wide.parquet), so we can
#      see which CHIRPS day lines up with the gauge day
# ============================================================
import ee

EE_PROJECT = "fine-blueprint-351008"      # <- edit: your Google Cloud project ID
DATASET = "UCSB-CHC/CHIRPS/V3/DAILY_RNL"
BAND = "precipitation"

# --- 1) initialize ---
try:
    ee.Initialize(project=EE_PROJECT)
except Exception as e:
    print("Initialize failed (", type(e).__name__, ") -> starting sign-in")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine initialized.")

# --- 2) dataset checks ---
col = ee.ImageCollection(DATASET)
n_period = col.filterDate("1983-01-01", "2024-01-01").size().getInfo()
img0 = col.filterDate("2021-07-07", "2021-07-08").first()
print("Bands:", img0.bandNames().getInfo())
print("Pixel size (m):", round(img0.select(BAND).projection().nominalScale().getInfo(), 0))
print("Daily images 1983-01-01 .. 2023-12-31:", n_period, "(expected 14975)")
print("Timestamp of the 2021-07-07 image:",
      ee.Date(img0.get("system:time_start")).format("YYYY-MM-dd HH:mm").getInfo(), "UTC")

# --- 3) test extraction ---
grid = pd.read_csv(SAT_DIR / "extraction_points.csv")
fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([r.lon, r.lat]), {"pt_id": r.pt_id})
    for r in grid.itertuples()
])

def sample_day(date_str):
    start = ee.Date(date_str)
    img = col.filterDate(start, start.advance(1, "day")).first().select(BAND)
    out = img.sampleRegions(collection=fc, scale=5566, geometries=False).getInfo()
    return {f["properties"]["pt_id"]: f["properties"].get(BAND) for f in out["features"]}

days = pd.date_range("2021-07-05", "2021-07-09")
res = {d: sample_day(d.strftime("%Y-%m-%d")) for d in days}
test = pd.DataFrame(res).T                      # rows = dates, columns = pt_id
print("\nPoints returned per day:", test.notna().sum(axis=1).to_dict(), "(expected 72)")

# --- 4) CHIRPS centre pixel vs gauge ---
centers = grid[grid["is_center"]].set_index("station")["pt_id"]
chirps_c = pd.DataFrame({st: test[pid] for st, pid in centers.items()}).round(1)
chirps_c.index.name = "date"
gauge = pd.read_parquet(PROCESSED_DIR / "rain_wide.parquet").loc["2021-07-05":"2021-07-09"]

print("\nCHIRPS centre-pixel value (mm), by date:")
print(chirps_c.to_string())
print("\nGauge value (mm), by date (Chapakot has no gauge):")
print(gauge.round(1).to_string())

Earth Engine initialized.
Bands: ['precipitation']
Pixel size (m): 5566.0
Daily images 1983-01-01 .. 2023-12-31: 14975 (expected 14975)
Timestamp of the 2021-07-07 image: 2021-07-07 00:00 UTC

Points returned per day: {Timestamp('2021-07-05 00:00:00'): 72, Timestamp('2021-07-06 00:00:00'): 72, Timestamp('2021-07-07 00:00:00'): 72, Timestamp('2021-07-08 00:00:00'): 72, Timestamp('2021-07-09 00:00:00'): 72} (expected 72)

CHIRPS centre-pixel value (mm), by date:
            Pokhara Airport  Kunchha  Bandipur  Lumle  Damauli  Sikles  Lamachaur  Chapakot
date                                                                                       
2021-07-05             14.2      9.5       7.3   18.6      8.6    12.0       15.2      20.4
2021-07-06             17.8     15.2       8.2   22.8      9.7    15.0       18.4       5.1
2021-07-07             59.6     37.4      29.5   80.3     23.7    51.0       71.3      46.1
2021-07-08             54.3     33.5      22.9   93.4     19.4    82.6     

In [5]:
# ============================================================
# CELL 3 (02_satellite_extraction): Full CHIRPS extraction, 1983-2023
# What it does:
#   1) for each year, stacks that year's daily images into one
#      multi-band image and samples all 72 points in a single request
#      (one band per day), then converts it to a date x point table
#   2) saves each year to data/satellite/chirps_v3_rnl_raw/ and skips
#      years that already exist (safe to rerun after an interruption);
#      retries a failed year up to 3 times
#   3) combines all years into data/satellite/chirps_v3_rnl_points.parquet
#      (rows = days, columns = pt_id) and prints shape, date range and
#      missing values
#   4) quick bias check: on days when the gauge has data, mean annual
#      rainfall from the CHIRPS centre pixel vs from the gauge, and the
#      CHIRPS/gauge ratio; Chapakot is shown for CHIRPS only
# ============================================================
import time

YEARS = range(1983, 2024)
RAW_DIR = SAT_DIR / "chirps_v3_rnl_raw"
RAW_DIR.mkdir(exist_ok=True)

def extract_year(y):
    imgs = (col.filterDate(f"{y}-01-01", f"{y + 1}-01-01")
               .sort("system:time_start").select(BAND))
    n = imgs.size().getInfo()
    stamps = imgs.aggregate_array("system:time_start").getInfo()
    names = [f"d{i:03d}" for i in range(n)]
    multi = imgs.toBands().rename(names)
    out = multi.sampleRegions(collection=fc, scale=5566, geometries=False).getInfo()
    rows = [[f["properties"]["pt_id"]] + [f["properties"].get(nm) for nm in names]
            for f in out["features"]]
    dates = pd.to_datetime(stamps, unit="ms").normalize()
    df = pd.DataFrame(rows, columns=["pt_id"] + list(dates)).set_index("pt_id").T
    df.index.name = "date"
    return df

t0 = time.time()
for y in YEARS:
    path = RAW_DIR / f"chirps_{y}.parquet"
    if path.exists():
        print(f"{y}: already saved, skipped")
        continue
    for attempt in range(1, 4):
        try:
            df_y = extract_year(y)
            break
        except Exception as e:
            print(f"{y}: attempt {attempt} failed ({type(e).__name__}: {str(e)[:120]})")
            time.sleep(5)
    else:
        raise RuntimeError(f"Year {y} failed 3 times; rerun the cell to resume.")
    df_y.to_parquet(path)
    print(f"{y}: {df_y.shape[0]} days x {df_y.shape[1]} points | elapsed {time.time() - t0:.0f}s")

# --- combine ---
chirps = pd.concat([pd.read_parquet(RAW_DIR / f"chirps_{y}.parquet") for y in YEARS]).sort_index()
chirps = chirps[grid["pt_id"].tolist()]                 # fixed column order
chirps.to_parquet(SAT_DIR / "chirps_v3_rnl_points.parquet")
print("\nCombined:", chirps.shape, "|", chirps.index.min().date(), "->", chirps.index.max().date())
print("Missing values:", int(chirps.isna().sum().sum()),
      "| duplicate dates:", int(chirps.index.duplicated().sum()),
      "| min:", round(float(chirps.min().min()), 2), "| max:", round(float(chirps.max().max()), 1))

# --- bias check at the centre pixel ---
rain_w = pd.read_parquet(PROCESSED_DIR / "rain_wide.parquet")
centers = grid[grid["is_center"]].set_index("station")["pt_id"]
rows = []
for st, pid in centers.items():
    if st == "Chapakot":
        rows.append({"station": st, "CHIRPS_mm_yr": round(chirps[pid].mean() * 365.25),
                     "gauge_mm_yr": np.nan, "ratio": np.nan})
        continue
    ok = rain_w[st].notna()
    c = chirps[pid][ok].mean() * 365.25
    g = rain_w[st][ok].mean() * 365.25
    rows.append({"station": st, "CHIRPS_mm_yr": round(c), "gauge_mm_yr": round(g),
                 "ratio": round(c / g, 2)})
print("\nMean annual rainfall on days with gauge data (centre pixel):")
print(pd.DataFrame(rows).to_string(index=False))

1983: already saved, skipped
1984: already saved, skipped
1985: already saved, skipped
1986: already saved, skipped
1987: already saved, skipped
1988: already saved, skipped
1989: already saved, skipped
1990: already saved, skipped
1991: already saved, skipped
1992: already saved, skipped
1993: already saved, skipped
1994: already saved, skipped
1995: already saved, skipped
1996: already saved, skipped
1997: already saved, skipped
1998: already saved, skipped
1999: already saved, skipped
2000: already saved, skipped
2001: already saved, skipped
2002: already saved, skipped
2003: already saved, skipped
2004: already saved, skipped
2005: already saved, skipped
2006: already saved, skipped
2007: already saved, skipped
2008: already saved, skipped
2009: already saved, skipped
2010: already saved, skipped
2011: already saved, skipped
2012: already saved, skipped
2013: already saved, skipped
2014: already saved, skipped
2015: already saved, skipped
2016: already saved, skipped
2017: already 

In [6]:
# ============================================================
# CELL 4 (02_satellite_extraction): How well does CHIRPS match the gauges?
# What it does:
#   1) alignment: for each gauge, correlates gauge day D with CHIRPS
#      centre pixel on D-1, D, D+1 and two blends that mimic the 08:45
#      reading time; prints all-year Spearman and monsoon Pearson
#      tables and picks the best candidate per station (monsoon Pearson)
#   2) at the best candidate: wet-day detection (POD, FAR, CSI at
#      >= 1 mm), wet-day frequency of CHIRPS vs gauge, and the
#      CHIRPS/gauge ratio by season
#   3) monthly check on months with a complete gauge record: monthly
#      Pearson r, median absolute % error and ratio for the centre pixel
#      versus the mean of its 8 surrounding pixels (a much better fit at
#      the centre pixel would hint that the gauge fed into CHIRPS)
# Needs chirps, grid, rain_wide.parquet and paths from earlier cells.
# ============================================================
SEASON_OF_MONTH = {12: "Winter", 1: "Winter", 2: "Winter",
                   3: "Pre-monsoon", 4: "Pre-monsoon", 5: "Pre-monsoon",
                   6: "Monsoon", 7: "Monsoon", 8: "Monsoon", 9: "Monsoon",
                   10: "Post-monsoon", 11: "Post-monsoon"}
SEASONS = ["Winter", "Pre-monsoon", "Monsoon", "Post-monsoon"]
WET = 1.0

rain_w = pd.read_parquet(PROCESSED_DIR / "rain_wide.parquet")
assert (rain_w.index == chirps.index).all(), "date indexes differ"

centers = grid[grid["is_center"]].set_index("station")["pt_id"]
ring = {st: grid[(grid["station"] == st) & (~grid["is_center"])]["pt_id"].tolist()
        for st in centers.index}
gauged = [s for s in centers.index if s != "Chapakot"]
season_s = pd.Series(rain_w.index.month.map(SEASON_OF_MONTH), index=rain_w.index)
mons = season_s == "Monsoon"

def candidates(c):
    return {
        "D-1": c.shift(1),
        "D": c,
        "D+1": c.shift(-1),
        "blend 21h D-1 + 3h D": 0.875 * c.shift(1) + 0.125 * c,
        "blend 21h D + 3h D+1": 0.875 * c + 0.125 * c.shift(-1),
    }

# --- 1) alignment ---
sp_rows, pm_rows, best = [], [], {}
for st in gauged:
    g = rain_w[st]
    cd = candidates(chirps[centers[st]])
    sp = {"station": st}
    pm = {"station": st}
    for name, s in cd.items():
        sp[name] = round(g.corr(s, method="spearman"), 3)
        pm[name] = round(g[mons].corr(s[mons]), 3)
    best[st] = max(cd, key=lambda n: pm[n])
    pm["best"] = best[st]
    sp_rows.append(sp)
    pm_rows.append(pm)

print("1a) Spearman correlation, all days (gauge vs CHIRPS candidate):")
print(pd.DataFrame(sp_rows).to_string(index=False))
print("\n1b) Pearson correlation, monsoon days only (best = highest):")
print(pd.DataFrame(pm_rows).to_string(index=False))

# --- 2) skill at the best candidate ---
rows, seas_rows = [], []
for st in gauged:
    g = rain_w[st]
    s = candidates(chirps[centers[st]])[best[st]]
    ok = g.notna() & s.notna()
    obs, sim = g[ok] >= WET, s[ok] >= WET
    hits = int((obs & sim).sum())
    miss = int((obs & ~sim).sum())
    fa = int((~obs & sim).sum())
    rows.append({
        "station": st, "candidate": best[st],
        "POD": round(hits / (hits + miss), 2),
        "FAR": round(fa / (hits + fa), 2),
        "CSI": round(hits / (hits + miss + fa), 2),
        "wet_freq_gauge_%": round(100 * obs.mean(), 1),
        "wet_freq_CHIRPS_%": round(100 * sim.mean(), 1),
    })
    ratio = {"station": st}
    for sea in SEASONS:
        m = ok & (season_s == sea)
        ratio[sea] = round(s[m].mean() / g[m].mean(), 2)
    seas_rows.append(ratio)

print("\n2a) Wet-day detection at the best candidate (>= 1 mm):")
print(pd.DataFrame(rows).to_string(index=False))
print("\n2b) CHIRPS / gauge mean rainfall ratio by season:")
print(pd.DataFrame(seas_rows).to_string(index=False))

# --- 3) monthly check: centre pixel vs surrounding ring ---
mper = rain_w.index.to_period("M")
rows = []
for st in gauged:
    cnt = rain_w[st].groupby(mper).count()
    n = rain_w[st].groupby(mper).size()
    full = cnt[cnt == n].index
    g_m = rain_w[st].groupby(mper).sum().loc[full]
    c_m = chirps[centers[st]].groupby(mper).sum().loc[full]
    r_m = chirps[ring[st]].mean(axis=1).groupby(mper).sum().loc[full]
    big = g_m >= 20
    rows.append({
        "station": st, "n_months": len(full),
        "r_centre": round(g_m.corr(c_m), 3), "r_ring": round(g_m.corr(r_m), 3),
        "medAPE_centre_%": round(100 * ((c_m - g_m).abs() / g_m)[big].median(), 1),
        "medAPE_ring_%": round(100 * ((r_m - g_m).abs() / g_m)[big].median(), 1),
        "ratio_centre": round(c_m.sum() / g_m.sum(), 2),
        "ratio_ring": round(r_m.sum() / g_m.sum(), 2),
    })
print("\n3) Monthly totals: gauge vs CHIRPS centre pixel and vs the mean of the 8 surrounding pixels:")
print(pd.DataFrame(rows).to_string(index=False))

1a) Spearman correlation, all days (gauge vs CHIRPS candidate):
        station   D-1     D   D+1  blend 21h D-1 + 3h D  blend 21h D + 3h D+1
Pokhara Airport 0.671 0.641 0.582                 0.680                 0.644
        Kunchha 0.509 0.506 0.464                 0.520                 0.509
       Bandipur 0.494 0.485 0.452                 0.506                 0.491
          Lumle 0.717 0.698 0.659                 0.726                 0.703
        Damauli 0.525 0.494 0.428                 0.535                 0.496
         Sikles 0.642 0.635 0.592                 0.652                 0.638
      Lamachaur 0.640 0.626 0.576                 0.649                 0.628

1b) Pearson correlation, monsoon days only (best = highest):
        station   D-1     D   D+1  blend 21h D-1 + 3h D  blend 21h D + 3h D+1                 best
Pokhara Airport 0.365 0.348 0.123                 0.388                 0.342 blend 21h D-1 + 3h D
        Kunchha 0.301 0.292 0.158                 0.

In [7]:
# ============================================================
# CELL 5 (02_satellite_extraction): Build and save the CHIRPS feature table
# What it does:
#   1) applies the chosen alignment to all 72 points: blend =
#      0.875 * CHIRPS(D-1) + 0.125 * CHIRPS(D), for gauge date D
#   2) for each of the 8 locations (7 gauges + Chapakot) builds features:
#        chirps_c_dm1 / chirps_c_d0 / chirps_c_dp1 : raw centre pixel on
#            D-1, D, D+1
#        chirps_blend_c : blend at the centre pixel
#        chirps_blend_mean9 / _max9 / _std9 : mean, max and spread of the
#            blend over the 3x3 block of pixels
#        chirps_blend_mean9_prev : mean9 on the previous gauge day
#        chirps_blend_mean9_sum3 : sum of mean9 over days D-2..D
#   3) stacks everything into one long table (station, date, features)
#   4) sanity check: monsoon Pearson correlation of the Pokhara centre
#      blend with its gauge should reproduce 0.388 from Cell 4
#   5) prints shape, missing values per column and mean annual mean9
#      per station, then saves data/satellite/chirps_features.parquet
# Note: the first row of each station has NaN for D-1 features (1982-12-31
# was not extracted); XGBoost handles NaN natively.
# ============================================================
blend_all = 0.875 * chirps.shift(1) + 0.125 * chirps     # all 72 points

frames = []
for st in centers.index:
    pid_c = centers[st]
    block = grid[grid["station"] == st]["pt_id"].tolist()   # the 9 points
    c = chirps[pid_c]
    b9 = blend_all[block]
    f = pd.DataFrame({
        "chirps_c_dm1": c.shift(1),
        "chirps_c_d0": c,
        "chirps_c_dp1": c.shift(-1),
        "chirps_blend_c": blend_all[pid_c],
        "chirps_blend_mean9": b9.mean(axis=1),
        "chirps_blend_max9": b9.max(axis=1),
        "chirps_blend_std9": b9.std(axis=1),
    })
    f["chirps_blend_mean9_prev"] = f["chirps_blend_mean9"].shift(1)
    f["chirps_blend_mean9_sum3"] = f["chirps_blend_mean9"].rolling(3, min_periods=3).sum()
    f.insert(0, "station", st)
    f.index.name = "date"
    frames.append(f.reset_index())

feat = pd.concat(frames, ignore_index=True)

# --- sanity check against Cell 4 ---
chk = feat[feat["station"] == "Pokhara Airport"].set_index("date")
m = chk.index.month.isin([6, 7, 8, 9])
r = rain_w["Pokhara Airport"][m].corr(chk["chirps_blend_c"][m])
print(f"Pokhara monsoon Pearson, chirps_blend_c vs gauge: {r:.3f} (expected 0.388)")

# --- report and save ---
print("\nFeature table:", feat.shape, "|", feat["date"].min().date(), "->", feat["date"].max().date())
print("\nMissing values per column:")
print(feat.drop(columns=["station", "date"]).isna().sum().to_string())
print("\nMean annual rainfall from chirps_blend_mean9 (mm/yr):")
print((feat.groupby("station")["chirps_blend_mean9"].mean() * 365.25).round(0).to_string())

feat.to_parquet(SAT_DIR / "chirps_features.parquet")
print("\nSaved: data/satellite/chirps_features.parquet")

Pokhara monsoon Pearson, chirps_blend_c vs gauge: 0.388 (expected 0.388)

Feature table: (119800, 11) | 1983-01-01 -> 2023-12-31

Missing values per column:
chirps_c_dm1                8
chirps_c_d0                 0
chirps_c_dp1                8
chirps_blend_c              8
chirps_blend_mean9          8
chirps_blend_max9           8
chirps_blend_std9           8
chirps_blend_mean9_prev    16
chirps_blend_mean9_sum3    24

Mean annual rainfall from chirps_blend_mean9 (mm/yr):
station
Bandipur           1744.0
Chapakot           1899.0
Damauli            1928.0
Kunchha            2462.0
Lamachaur          3867.0
Lumle              4083.0
Pokhara Airport    3623.0
Sikles             3339.0

Saved: data/satellite/chirps_features.parquet


In [8]:
# ============================================================
# CELL 6 (02_satellite_extraction): ERA5-Land hourly - band check and window test
# What it does:
#   1) loads ECMWF/ERA5_LAND/HOURLY, lists the bands containing
#      'precipitation', 'temperature', 'wind' and 'pressure', and stops
#      with a clear message if the hourly precipitation band is missing
#   2) pulls hourly precipitation (converted to mm) at the centre pixel
#      of Pokhara Airport, Lumle and Chapakot for 2021-07-05 00:00 to
#      2021-07-08 13:00 UTC
#   3) for gauge days 2021-07-06, 07-07 and 07-08, sums 24 hourly values
#      ending at 00, 03, 06, 09 and 12 UTC (03 UTC = 08:45 Nepal time =
#      the gauge reading time); assumes each image is stamped at the END
#      of the hour it covers
#   4) prints those window totals next to the gauge value and the CHIRPS
#      blend for the same days
# Needs ee (initialized), pts, rain_wide (rain_w) and feat from earlier cells.
# ============================================================
ERA_DATASET = "ECMWF/ERA5_LAND/HOURLY"
era = ee.ImageCollection(ERA_DATASET)

# --- 1) band check ---
first = era.filterDate("2021-07-07", "2021-07-08").first()
names = first.bandNames().getInfo()
for key in ("precipitation", "temperature", "wind", "pressure"):
    print(f"Bands containing '{key}':", [n for n in names if key in n])
print("Pixel size (m):", round(first.select(names[0]).projection().nominalScale().getInfo(), 0))

P_BAND = "total_precipitation_hourly"
if P_BAND not in names:
    raise ValueError(f"'{P_BAND}' not found; choose the hourly precipitation band from the list above")

# --- 2) hourly precipitation at three centre pixels ---
T0, T1 = "2021-07-05T00:00:00", "2021-07-08T13:00:00"
test_locs = ["Pokhara Airport", "Lumle", "Chapakot"]
hourly = {}
for st in test_locs:
    geom = ee.Geometry.Point([float(pts.loc[st, "lon"]), float(pts.loc[st, "lat"])])
    data = (era.select(P_BAND).filterDate(ee.Date(T0), ee.Date(T1))
               .getRegion(geom, 11132).getInfo())
    d = pd.DataFrame(data[1:], columns=data[0])
    d["time"] = pd.to_datetime(d["time"], unit="ms")
    hourly[st] = d.set_index("time")[P_BAND].astype(float) * 1000     # m -> mm
    print(f"{st}: {len(d)} hourly values, {d['time'].min()} -> {d['time'].max()}")

# --- 3) 24-hour window totals for different end hours ---
dates = pd.to_datetime(["2021-07-06", "2021-07-07", "2021-07-08"])
ends = [0, 3, 6, 9, 12]
rows = []
for st, s in hourly.items():
    for d in dates:
        r = {"location": st, "gauge_day": d.date()}
        for h in ends:
            end = d + pd.Timedelta(hours=h)
            w = s.loc[end - pd.Timedelta(hours=23): end]
            r[f"end {h:02d}Z"] = round(float(w.sum()), 1) if len(w) == 24 else np.nan
        rows.append(r)
tab = pd.DataFrame(rows)

# --- 4) context: gauge and CHIRPS blend ---
ctx = []
for st in test_locs:
    for d in dates:
        g = rain_w.loc[d, st] if st in rain_w.columns else np.nan
        cb = feat[(feat["station"] == st) & (feat["date"] == d)]["chirps_blend_c"].iloc[0]
        ctx.append({"location": st, "gauge_day": d.date(),
                    "gauge_mm": round(float(g), 1) if pd.notna(g) else np.nan,
                    "CHIRPS_blend": round(float(cb), 1)})
out = tab.merge(pd.DataFrame(ctx), on=["location", "gauge_day"])
print("\nERA5-Land 24 h totals (mm) by window end hour (UTC), next to gauge and CHIRPS:")
print(out.to_string(index=False))

Bands containing 'precipitation': ['total_precipitation', 'total_precipitation_hourly']
Bands containing 'temperature': ['dewpoint_temperature_2m', 'temperature_2m', 'skin_temperature', 'soil_temperature_level_1', 'soil_temperature_level_2', 'soil_temperature_level_3', 'soil_temperature_level_4', 'lake_bottom_temperature', 'lake_ice_temperature', 'lake_mix_layer_temperature', 'lake_total_layer_temperature', 'temperature_of_snow_layer']
Bands containing 'wind': ['u_component_of_wind_10m', 'v_component_of_wind_10m']
Bands containing 'pressure': ['surface_pressure']
Pixel size (m): 11132.0
Pokhara Airport: 85 hourly values, 2021-07-05 00:00:00 -> 2021-07-08 12:00:00
Lumle: 85 hourly values, 2021-07-05 00:00:00 -> 2021-07-08 12:00:00
Chapakot: 85 hourly values, 2021-07-05 00:00:00 -> 2021-07-08 12:00:00

ERA5-Land 24 h totals (mm) by window end hour (UTC), next to gauge and CHIRPS:
       location  gauge_day  end 00Z  end 03Z  end 06Z  end 09Z  end 12Z  gauge_mm  CHIRPS_blend
Pokhara Airpo

In [9]:
# ============================================================
# CELL 7 (02_satellite_extraction): ERA5-Land window alignment + independence check
# What it does:
#   1) downloads hourly ERA5-Land precipitation (mm) at the centre pixel
#      of the 7 gauged stations for 2006-2012, one file per station-year
#      in data/satellite/era5land_hourly_precip/ (skips existing files;
#      a failing request is split in halves and retried)
#   2) builds 24-hour rolling totals (stamp = end of window) and, for each
#      gauge date D, takes the window ending at D 00:00 UTC + o hours for
#      o = -21, -18, ..., +48
#   3) monsoon Pearson correlation of each window with (a) the gauge
#      value of day D and (b) CHIRPS centre pixel on day D; prints the
#      full table and the best end hour per station
#   4) independence check: CHIRPS centre pixel vs ERA5-Land UTC-day total
#      (window ending at D+1 00:00 UTC): monsoon Pearson, all-days
#      Spearman and the mean ratio; also the gauge vs the same ERA5-Land
#      totals for reference
# Needs ee, era, P_BAND, pts, centers, chirps and rain_w from earlier cells.
# ============================================================
import time

ERA_DIR = SAT_DIR / "era5land_hourly_precip"
ERA_DIR.mkdir(exist_ok=True)
YEARS_T = range(2006, 2013)
test_stations = [s for s in centers.index if s != "Chapakot"]

def fetch_hourly(geom, t0, t1):
    try:
        data = (era.select(P_BAND).filterDate(ee.Date(t0), ee.Date(t1))
                   .getRegion(geom, 11132).getInfo())
    except Exception:
        span = pd.Timestamp(t1) - pd.Timestamp(t0)
        if span <= pd.Timedelta(days=40):
            raise
        mid = (pd.Timestamp(t0) + span / 2).floor("h").strftime("%Y-%m-%dT%H:%M:%S")
        return pd.concat([fetch_hourly(geom, t0, mid), fetch_hourly(geom, mid, t1)])
    d = pd.DataFrame(data[1:], columns=data[0])
    d["time"] = pd.to_datetime(d["time"], unit="ms")
    return d.set_index("time")[P_BAND].astype(float).mul(1000).sort_index()

# --- 1) download ---
t_start = time.time()
hourly_all = {}
for st in test_stations:
    geom = ee.Geometry.Point([float(pts.loc[st, "lon"]), float(pts.loc[st, "lat"])])
    parts = []
    for y in YEARS_T:
        path = ERA_DIR / f"{st.replace(' ', '_')}_{y}.parquet"
        if path.exists():
            parts.append(pd.read_parquet(path)["mm"])
            continue
        s = fetch_hourly(geom, f"{y}-01-01T00:00:00", f"{y + 1}-01-01T01:00:00").rename("mm")
        s.to_frame().to_parquet(path)
        parts.append(s)
    h = pd.concat(parts).sort_index()
    hourly_all[st] = h[~h.index.duplicated(keep="first")]
    print(f"{st}: {len(h)} hourly values | elapsed {time.time() - t_start:.0f}s")

# --- 2-3) window scan ---
offsets = list(range(-21, 49, 3))
days_test = pd.date_range("2006-01-01", "2012-12-31")
mons_test = days_test.month.isin([6, 7, 8, 9])

s24_all, res_g, res_c = {}, {}, {}
for st in test_stations:
    s = hourly_all[st]
    s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq="h"))
    s24 = s.rolling(24, min_periods=24).sum()
    s24_all[st] = s24
    g = rain_w[st].reindex(days_test)
    c = chirps[centers[st]].reindex(days_test)
    rg, rc = {}, {}
    for o in offsets:
        v = pd.Series(s24.reindex(days_test + pd.Timedelta(hours=o)).values, index=days_test)
        rg[o] = g[mons_test].corr(v[mons_test])
        rc[o] = c[mons_test].corr(v[mons_test])
    res_g[st], res_c[st] = pd.Series(rg), pd.Series(rc)

G, C = pd.DataFrame(res_g), pd.DataFrame(res_c)
G.index.name = C.index.name = "end_h"
print("\n1) Monsoon Pearson: GAUGE day D vs ERA5-Land 24 h total ending at D 00 UTC + end_h hours")
print(G.round(3).to_string())
print("\n2) Monsoon Pearson: CHIRPS day D vs the same ERA5-Land windows")
print(C.round(3).to_string())

summ = pd.DataFrame({
    "gauge_best_end_h": G.idxmax(), "gauge_r_best": G.max().round(3),
    "gauge_r_at_+3": G.loc[3].round(3),
    "CHIRPS_best_end_h": C.idxmax(), "CHIRPS_r_best": C.max().round(3),
    "CHIRPS_r_at_+24": C.loc[24].round(3),
})
print("\n3) Best end hour per station:")
print(summ.to_string())

# --- 4) independence: CHIRPS vs ERA5-Land UTC-day totals ---
rows = []
for st in test_stations:
    v = pd.Series(s24_all[st].reindex(days_test + pd.Timedelta(hours=24)).values, index=days_test)
    c = chirps[centers[st]].reindex(days_test)
    g = rain_w[st].reindex(days_test)
    rows.append({
        "station": st,
        "CHIRPS_vs_ERA5L_pearson_monsoon": round(c[mons_test].corr(v[mons_test]), 3),
        "CHIRPS_vs_ERA5L_spearman_all": round(c.corr(v, method="spearman"), 3),
        "mean_ratio_CHIRPS/ERA5L": round(c.mean() / v.mean(), 2),
        "gauge_vs_ERA5L_pearson_monsoon": round(g[mons_test].corr(v[mons_test]), 3),
    })
print("\n4) CHIRPS vs ERA5-Land UTC-day totals (window ending D+1 00 UTC):")
print(pd.DataFrame(rows).to_string(index=False))

Pokhara Airport: 61375 hourly values | elapsed 0s
Kunchha: 61375 hourly values | elapsed 0s
Bandipur: 61375 hourly values | elapsed 0s
Lumle: 61375 hourly values | elapsed 0s
Damauli: 61375 hourly values | elapsed 1s
Sikles: 61375 hourly values | elapsed 1s
Lamachaur: 61375 hourly values | elapsed 1s

1) Monsoon Pearson: GAUGE day D vs ERA5-Land 24 h total ending at D 00 UTC + end_h hours
       Pokhara Airport  Kunchha  Bandipur  Lumle  Damauli  Sikles  Lamachaur
end_h                                                                       
-21              0.196    0.150     0.214  0.122    0.169   0.119      0.116
-18              0.206    0.148     0.219  0.125    0.182   0.113      0.125
-15              0.234    0.154     0.218  0.139    0.205   0.115      0.145
-12              0.252    0.157     0.215  0.150    0.219   0.123      0.158
-9               0.273    0.167     0.216  0.161    0.233   0.132      0.175
-6               0.311    0.188     0.232  0.183    0.256   0.141    

In [10]:
# ============================================================
# CELL 8 (02_satellite_extraction): Is Sikles' date label shifted? (4 time blocks)
# What it does:
#   A) satellite check, monsoon Pearson per block: for each station,
#      r(gauge D vs CHIRPS D-1) minus r(gauge D vs CHIRPS D+1). Aligned
#      stations score clearly positive; a station whose date label is a
#      day early scores near zero or negative
#   B) gauge-only check: for each station, correlation of the reference
#      (mean of Pokhara Airport, Lamachaur and Lumle, excluding the
#      station itself) on day t with the station on day t-1 minus the same
#      with the station on day t+1. About 0 = aligned; clearly positive =
#      the station's date label is earlier than the reference's
#   C) Sikles detail: the three correlations behind A and B for each block
# Blocks: 1983-1992, 1993-2002, 2003-2012, 2013-2023, plus the full period
# Needs rain_w, chirps, centers and test_stations from earlier cells.
# ============================================================
blocks = [("1983-1992", 1983, 1992), ("1993-2002", 1993, 2002),
          ("2003-2012", 2003, 2012), ("2013-2023", 2013, 2023),
          ("full 1983-2023", 1983, 2023)]
yrs = rain_w.index.year
mon_mask = rain_w.index.month.isin([6, 7, 8, 9])
refset = ["Pokhara Airport", "Lamachaur", "Lumle"]

def block_mask(y0, y1):
    return mon_mask & (yrs >= y0) & (yrs <= y1)

rowsA, rowsB, rowsC = [], [], []
for lab, y0, y1 in blocks:
    m = block_mask(y0, y1)
    ra, rb = {"block": lab}, {"block": lab}
    for st in test_stations:
        g = rain_w[st]
        c = chirps[centers[st]]
        # A) CHIRPS: D-1 minus D+1
        a_m1 = g[m].corr(c.shift(1)[m])
        a_p1 = g[m].corr(c.shift(-1)[m])
        ra[st] = round(a_m1 - a_p1, 3)
        # B) gauge-only: reference(t) vs station(t-1) minus station(t+1)
        refs = [x for x in refset if x != st]
        ref = rain_w[refs].mean(axis=1, skipna=False)
        b_m1 = ref[m].corr(g.shift(1)[m])
        b_p1 = ref[m].corr(g.shift(-1)[m])
        rb[st] = round(b_m1 - b_p1, 3)
        if st == "Sikles":
            rowsC.append({
                "block": lab,
                "CHIRPS r(D-1)": round(a_m1, 3),
                "CHIRPS r(D)": round(g[m].corr(c[m]), 3),
                "CHIRPS r(D+1)": round(a_p1, 3),
                "ref r(Sikles t-1)": round(b_m1, 3),
                "ref r(Sikles t)": round(ref[m].corr(g[m]), 3),
                "ref r(Sikles t+1)": round(b_p1, 3),
            })
    rowsA.append(ra)
    rowsB.append(rb)

print("A) CHIRPS check: monsoon r(D-1) minus r(D+1)  (aligned stations: clearly positive)")
print(pd.DataFrame(rowsA).to_string(index=False))
print("\nB) Gauge-only check: ref(t) vs station(t-1) minus vs station(t+1)  (aligned: about 0; "
      "station label earlier than reference: positive)")
print(pd.DataFrame(rowsB).to_string(index=False))
print("\nC) Sikles detail by block (monsoon Pearson):")
print(pd.DataFrame(rowsC).to_string(index=False))

A) CHIRPS check: monsoon r(D-1) minus r(D+1)  (aligned stations: clearly positive)
         block  Pokhara Airport  Kunchha  Bandipur  Lumle  Damauli  Sikles  Lamachaur
     1983-1992            0.227    0.149     0.048  0.051    0.247   0.060      0.087
     1993-2002            0.280    0.123     0.113  0.220    0.158   0.111      0.208
     2003-2012            0.253    0.180     0.094  0.071    0.388  -0.127      0.179
     2013-2023            0.216    0.125     0.290  0.177    0.271   0.062      0.109
full 1983-2023            0.242    0.143     0.137  0.134    0.264   0.030      0.143

B) Gauge-only check: ref(t) vs station(t-1) minus vs station(t+1)  (aligned: about 0; station label earlier than reference: positive)
         block  Pokhara Airport  Kunchha  Bandipur  Lumle  Damauli  Sikles  Lamachaur
     1983-1992            0.010   -0.006    -0.115  0.009   -0.119  -0.014     -0.021
     1993-2002            0.056    0.011     0.024 -0.066    0.020  -0.041      0.016
     200

In [11]:
# ============================================================
# CELL 9 (02_satellite_extraction): Year-by-year date-label check
# What it does:
#   1) for each year 1983-2023, in Jun-Sep only, correlates the reference
#      (mean of Pokhara Airport, Lamachaur, Lumle, excluding the station
#      itself) on day t with the station on day t-1, t and t+1
#      'best' = tm1 means the station's value filed on day t-1 matches the
#      reference on day t best, i.e. its date label is one day early
#   2) the same year, CHIRPS check: r(gauge D, CHIRPS D+1) minus
#      r(gauge D, CHIRPS D-1); negative = label early (aligned stations
#      are clearly positive)
#   3) prints the full table for Sikles (41 rows, flagged years marked)
#   4) noise control: for every station, how many years get best = tm1
#      or tp1, and the mean margin, so we can judge how many flagged
#      years at Sikles go beyond chance
# Needs rain_w, chirps, centers, test_stations, refset, mon_mask, yrs.
# ============================================================
def year_table(st, y0=1983, y1=2023, min_n=60):
    refs = [x for x in refset if x != st]
    ref = rain_w[refs].mean(axis=1, skipna=False)
    g = rain_w[st]
    c = chirps[centers[st]]
    rows = []
    for y in range(y0, y1 + 1):
        m = mon_mask & (yrs == y)
        n = int((ref[m].notna() & g[m].notna()).sum())
        if n < min_n:
            rows.append({"year": y, "n": n, "r_tm1": np.nan, "r_t": np.nan, "r_tp1": np.nan,
                         "margin": np.nan, "best": "n/a", "chirps_A": np.nan})
            continue
        r = {"tm1": ref[m].corr(g.shift(1)[m]),
             "t": ref[m].corr(g[m]),
             "tp1": ref[m].corr(g.shift(-1)[m])}
        best = max(r, key=lambda k: r[k])
        A = g[m].corr(c.shift(-1)[m]) - g[m].corr(c.shift(1)[m])    # r(D+1) - r(D-1)
        rows.append({"year": y, "n": n,
                     "r_tm1": round(r["tm1"], 3), "r_t": round(r["t"], 3), "r_tp1": round(r["tp1"], 3),
                     "margin": round(r["tm1"] - r["t"], 3),
                     "best": best, "chirps_A": round(-A, 3)})
    out = pd.DataFrame(rows).set_index("year")
    return out

sk = year_table("Sikles")
sk["flag"] = np.where(sk["best"] == "tm1", "<-- early", "")
print("Sikles, per year (Jun-Sep). best=tm1: date label one day early;")
print("chirps_A is r(D-1) minus r(D+1) (aligned: positive, early: negative); margin = r_tm1 - r_t")
print(sk.to_string())

# --- noise control ---
rows = []
for st in test_stations:
    t = sk if st == "Sikles" else year_table(st)
    used = t[t["best"] != "n/a"]
    rows.append({
        "station": st,
        "years_used": len(used),
        "best_tm1": int((used["best"] == "tm1").sum()),
        "best_t": int((used["best"] == "t").sum()),
        "best_tp1": int((used["best"] == "tp1").sum()),
        "mean_margin": round(used["margin"].mean(), 3),
        "years_chirps_A<0": int((used["chirps_A"] < 0).sum()),
    })
print("\nNoise control: how often each station gets flagged (best = tm1) or shows chirps_A < 0:")
print(pd.DataFrame(rows).to_string(index=False))

Sikles, per year (Jun-Sep). best=tm1: date label one day early;
chirps_A is r(D-1) minus r(D+1) (aligned: positive, early: negative); margin = r_tm1 - r_t
        n  r_tm1    r_t  r_tp1  margin best  chirps_A       flag
year                                                            
1983  122  0.110  0.443  0.153  -0.333    t     0.037           
1984  122  0.059  0.489  0.058  -0.431    t     0.005           
1985  122  0.216  0.430  0.321  -0.214    t     0.035           
1986  122  0.144  0.540  0.116  -0.395    t     0.062           
1987  122  0.124  0.563  0.229  -0.439    t     0.013           
1988  122  0.193  0.446  0.283  -0.252    t     0.317           
1989  122  0.120  0.381 -0.083  -0.261    t    -0.027           
1990  122  0.255  0.551  0.078  -0.296    t    -0.057           
1991  122 -0.096  0.269  0.121  -0.365    t     0.139           
1992  122  0.062  0.625  0.154  -0.563    t     0.202           
1993  122  0.090  0.612  0.272  -0.522    t    -0.028           


In [12]:
# ============================================================
# CELL 10 (02_satellite_extraction): Pin down the start/end of Sikles' shifted periods
# What it does:
#   for Sikles (and Lumle as a control), for each calendar month of the
#   selected years:
#     - reference = mean of the other stations in {Pokhara Airport,
#       Lamachaur, Lumle}, on days when at least 2 of them have data
#       (so 2005 can be tested; for Sikles that is 3 stations, for Lumle 2)
#     - margin = Spearman r(reference t, station t-1) minus
#                Spearman r(reference t, station t)
#       positive = station's date label early (shifted), negative = aligned
#     - months with < 20 valid days or < 6 reference-wet days (>= 1 mm)
#       are left blank
#   prints one table per station: rows = years, columns = months 1-12
# Years shown: 2003-2007, 2009-2012, 2018-2023 (the transition windows)
# Needs rain_w and refset from earlier cells.
# ============================================================
YEARS_SHOW = [2003, 2004, 2005, 2006, 2007, 2009, 2010, 2011, 2012,
              2018, 2019, 2020, 2021, 2022, 2023]

def margin_table(st, years, min_valid=20, min_wet=6):
    refs = [x for x in refset if x != st]
    r3 = rain_w[refs]
    ref = r3.mean(axis=1).where(r3.notna().sum(axis=1) >= 2)
    g = rain_w[st]
    g_prev = g.shift(1)
    out = {}
    for y in years:
        for m in range(1, 13):
            idx = (rain_w.index.year == y) & (rain_w.index.month == m)
            rr = ref[idx]
            ok = rr.notna() & g[idx].notna() & g_prev[idx].notna()
            if ok.sum() < min_valid or (rr[ok] >= 1).sum() < min_wet:
                out[(y, m)] = np.nan
                continue
            r_prev = rr[ok].corr(g_prev[idx][ok], method="spearman")
            r_same = rr[ok].corr(g[idx][ok], method="spearman")
            out[(y, m)] = r_prev - r_same
    s = pd.Series(out)
    tab = s.unstack().reindex(columns=range(1, 13))
    tab.index.name = "year"
    return tab

for st in ["Sikles", "Lumle"]:
    t = margin_table(st, YEARS_SHOW).round(2)
    print(f"\n{st}: margin = r(ref t, {st} t-1) - r(ref t, {st} t); positive = label early (shifted)")
    print("columns = month 1..12; blank = not enough rain days to test")
    print(t.astype(object).where(t.notna(), "").to_string())


Sikles: margin = r(ref t, Sikles t-1) - r(ref t, Sikles t); positive = label early (shifted)
columns = month 1..12; blank = not enough rain days to test
        1     2     3     4     5     6     7     8     9     10    11    12
year                                                                        
2003             -0.08  0.29 -0.32  0.21  0.09 -0.47  0.21 -0.27       -0.12
2004       -0.77 -0.19 -0.46  0.12 -0.08 -0.52 -0.21 -0.39 -0.38            
2005 -0.04        0.25  0.05  0.24  0.18  0.61 -0.45 -0.04  0.32            
2006              0.38  0.09  0.29  0.48 -0.09  0.38  0.26  0.17            
2007        0.18  0.33 -0.02  0.37  0.24  0.78  0.28  0.42  0.38  0.32      
2009             -0.03  0.27  1.01  0.14  0.35  0.73  0.15  0.09            
2010              0.17  0.15 -0.18  0.19  0.32  0.49  0.24 -0.05            
2011              0.45  0.24  0.12  0.14 -0.38 -0.61 -0.18  0.16  0.19      
2012                   -0.07 -0.09 -0.05  0.05  -0.7  0.16 -0.33            

In [13]:
# ============================================================
# CELL 11 (02_satellite_extraction): Candidate correction for Sikles (nothing is saved)
# What it does:
#   1) defines apply_shift(): for a shifted range [A, B], the value stored
#      on date d (A <= d <= B) is moved to d+1; date A and date B+1 are
#      set to NaN (the boundary days are ambiguous); at the end of the
#      data the last stored value is dropped
#   2) boundary scans: for each transition, tries several candidate dates
#      and prints the Spearman/Pearson correlation between the reference
#      (mean of Pokhara Airport, Lamachaur, Lumle with >= 2 stations) and
#      Sikles in a window around the transition; the 'original' row is
#      the uncorrected series. Highest correlation = best candidate
#   3) before/after table for 4 periods (monsoon days): Sikles vs
#      reference, Sikles vs CHIRPS blend, and a placebo with Lumle
#      (same shift, Lumle vs its reference of Pokhara + Lamachaur)
#   4) summary: how many days are shifted and how many become NaN
# Defaults used for the before/after table (edit if a scan disagrees):
#   S1 = 2005-02-01 .. 2011-06-30 ; S2 = 2020-02-01 .. 2023-12-31
# Needs rain_w, feat, mon_mask from earlier cells.
# ============================================================
A1, B1, A2, END = "2005-02-01", "2011-06-30", "2020-02-01", "2023-12-31"
DAY = pd.Timedelta(days=1)

def apply_shift(s, ranges):
    new = s.copy()
    for a, b in ranges:
        a, b = pd.Timestamp(a), pd.Timestamp(b)
        seg = s.loc[a:b]                                  # values labelled a..b
        new.loc[a] = np.nan
        new.loc[a + DAY: b] = seg.iloc[:-1].values        # value of d moves to d+1
        if (b + DAY) in new.index:
            new.loc[b + DAY] = np.nan
    return new

sik = rain_w["Sikles"]
lum = rain_w["Lumle"]
refS3 = rain_w[["Pokhara Airport", "Lamachaur", "Lumle"]]
refS = refS3.mean(axis=1).where(refS3.notna().sum(axis=1) >= 2)
refL3 = rain_w[["Pokhara Airport", "Lamachaur"]]
refL = refL3.mean(axis=1).where(refL3.notna().sum(axis=1) >= 2)

# --- 2) boundary scans ---
def scan(cands, window, make_ranges):
    w = slice(*window)
    rows = []
    for c in ["original"] + cands:
        s = sik if c == "original" else apply_shift(sik, make_ranges(c))
        x, y = refS.loc[w], s.loc[w]
        ok = x.notna() & y.notna()
        rows.append({"candidate": c, "n_days": int(ok.sum()),
                     "spearman": round(x[ok].corr(y[ok], method="spearman"), 3),
                     "pearson": round(x[ok].corr(y[ok]), 3)})
    return pd.DataFrame(rows)

print("2a) Start of S1: window Nov 2004 - Apr 2005 (candidate = first shifted day)")
print(scan(["2004-11-01", "2004-12-01", "2005-01-01", "2005-02-01", "2005-03-01"],
           ("2004-11-01", "2005-04-30"), lambda c: [(c, B1), (A2, END)]).to_string(index=False))
print("\n2b) End of S1: window Mar - Sep 2011 (candidate = last shifted day)")
print(scan(["2011-04-30", "2011-05-31", "2011-06-30", "2011-07-31"],
           ("2011-03-01", "2011-09-30"), lambda c: [(A1, c), (A2, END)]).to_string(index=False))
print("\n2c) Start of S2: window Nov 2019 - Apr 2020 (candidate = first shifted day)")
print(scan(["2019-11-01", "2019-12-01", "2020-01-01", "2020-02-01", "2020-03-01"],
           ("2019-11-01", "2020-04-30"), lambda c: [(A1, B1), (c, END)]).to_string(index=False))

# --- 3) before / after ---
ranges = [(A1, B1), (A2, END)]
sik_c = apply_shift(sik, ranges)
lum_c = apply_shift(lum, ranges)
chirps_s = feat[feat["station"] == "Sikles"].set_index("date")["chirps_blend_c"]

def pear(x, y, m):
    ok = x[m].notna() & y[m].notna()
    if ok.sum() < 30:
        return np.nan
    return round(x[m][ok].corr(y[m][ok]), 3)

periods = [("1983-01..2005-01 (aligned)", "1983-01-01", "2005-01-31"),
           ("2005-02..2011-06 (S1)", A1, B1),
           ("2011-07..2020-01 (aligned)", "2011-07-01", "2020-01-31"),
           ("2020-02..2023-12 (S2)", A2, END)]
rows = []
for lab, a, b in periods:
    m = (rain_w.index >= pd.Timestamp(a)) & (rain_w.index <= pd.Timestamp(b)) & mon_mask
    rows.append({
        "period (Jun-Sep days)": lab,
        "Sik~ref orig": pear(sik, refS, m), "Sik~ref corr": pear(sik_c, refS, m),
        "Sik~CHIRPS orig": pear(sik, chirps_s, m), "Sik~CHIRPS corr": pear(sik_c, chirps_s, m),
        "Lumle~ref orig": pear(lum, refL, m), "Lumle~ref corr (placebo)": pear(lum_c, refL, m),
    })
print("\n3) Monsoon Pearson correlation before and after the candidate correction:")
print(pd.DataFrame(rows).to_string(index=False))

# --- 4) summary ---
shifted_days = (pd.Timestamp(B1) - pd.Timestamp(A1)).days + (pd.Timestamp(END) - pd.Timestamp(A2)).days + 2
new_nan = int(sik_c.isna().sum() - sik.isna().sum())
print(f"\n4) Days in the shifted ranges: {shifted_days} of {len(sik)} "
      f"({100 * shifted_days / len(sik):.1f}% of the record); extra NaN days created: {new_nan}")

2a) Start of S1: window Nov 2004 - Apr 2005 (candidate = first shifted day)
 candidate  n_days  spearman  pearson
  original     181     0.476    0.257
2004-11-01     180     0.583    0.310
2004-12-01     180     0.621    0.357
2005-01-01     180     0.600    0.357
2005-02-01     180     0.615    0.340
2005-03-01     180     0.561    0.333

2b) End of S1: window Mar - Sep 2011 (candidate = last shifted day)
 candidate  n_days  spearman  pearson
  original     214     0.615    0.567
2011-04-30     213     0.655    0.574
2011-05-31     213     0.668    0.575
2011-06-30     213     0.665    0.549
2011-07-31     213     0.641    0.517

2c) Start of S2: window Nov 2019 - Apr 2020 (candidate = first shifted day)
 candidate  n_days  spearman  pearson
  original     182     0.562    0.447
2019-11-01     181     0.630    0.504
2019-12-01     181     0.657    0.508
2020-01-01     181     0.685    0.590
2020-02-01     181     0.697    0.677
2020-03-01     181     0.687    0.661

3) Monsoon Pearso

In [14]:
# ============================================================
# CELL 12 (02_satellite_extraction): Apply the Sikles date correction to a NEW file
# What it does:
#   1) applies apply_shift() to Sikles with the final ranges
#        S1 = 2005-01-01 .. 2011-05-31 ; S2 = 2020-02-01 .. 2023-12-31
#      (value stored on date d moves to d+1; the first day of each range and
#      the day after S1 are set to NaN) and asserts that the other six
#      stations are identical to the originals
#   2) saves data/processed/rain_wide_aligned.parquet and a log of the
#      correction, data/processed/date_fixes.json
#   3) checks: Sikles monsoon Pearson vs the CHIRPS blend, original vs
#      corrected, by period; and how many years of 1983-2023 have best
#      day = t-1 (label early) for Sikles before and after, using the
#      reference mean of Pokhara Airport, Lamachaur and Lumle
#   4) lists the remaining flagged years after correction
# Needs apply_shift, rain_w, feat, mon_mask, yrs, refset, PROCESSED_DIR.
# ============================================================
import json

A1, B1, A2, END = "2005-01-01", "2011-05-31", "2020-02-01", "2023-12-31"
SHIFT_RANGES = [(A1, B1), (A2, END)]

aligned = rain_w.copy()
aligned["Sikles"] = apply_shift(rain_w["Sikles"], SHIFT_RANGES)

others = [c for c in rain_w.columns if c != "Sikles"]
assert aligned[others].equals(rain_w[others]), "other stations changed unexpectedly"
print("Other six stations unchanged: True")

new_nan = int(aligned["Sikles"].isna().sum() - rain_w["Sikles"].isna().sum())
shifted_days = sum((pd.Timestamp(b) - pd.Timestamp(a)).days + 1 for a, b in SHIFT_RANGES)
print(f"Sikles: {shifted_days} days in shifted ranges; extra NaN days created: {new_nan}")

# --- 2) save ---
aligned.to_parquet(PROCESSED_DIR / "rain_wide_aligned.parquet")
log = {
    "station": "Sikles",
    "issue": "date label one day early (value stored on d belongs to d+1) in two periods",
    "shifted_ranges": [{"first_label": a, "last_label": b} for a, b in SHIFT_RANGES],
    "treatment": "value stored on d moved to d+1; first day of each range and the day after the first range set to NaN",
    "evidence": ["reference-gauge correlation (Pokhara Airport, Lamachaur, Lumle)",
                 "CHIRPS correlation", "placebo with Lumle"],
    "caveat": "boundaries chosen on the same data used to score them; cause not yet traced to the source files",
}
with open(PROCESSED_DIR / "date_fixes.json", "w") as f:
    json.dump(log, f, indent=2)
print("Saved: rain_wide_aligned.parquet, date_fixes.json")

# --- 3a) Sikles vs CHIRPS blend by period ---
chirps_s = feat[feat["station"] == "Sikles"].set_index("date")["chirps_blend_c"]
def pear(x, y, m):
    ok = x[m].notna() & y[m].notna()
    return round(x[m][ok].corr(y[m][ok]), 3) if ok.sum() >= 30 else np.nan

periods = [("1983-01..2004-12", "1983-01-01", "2004-12-31"),
           ("2005-01..2011-05 (S1)", A1, B1),
           ("2011-06..2020-01", "2011-06-01", "2020-01-31"),
           ("2020-02..2023-12 (S2)", A2, END),
           ("full 1983-2023", "1983-01-01", END)]
rows = []
for lab, a, b in periods:
    m = (rain_w.index >= pd.Timestamp(a)) & (rain_w.index <= pd.Timestamp(b)) & mon_mask
    rows.append({"period (Jun-Sep days)": lab,
                 "Sikles~CHIRPS orig": pear(rain_w["Sikles"], chirps_s, m),
                 "Sikles~CHIRPS corrected": pear(aligned["Sikles"], chirps_s, m)})
print("\n3a) Sikles vs CHIRPS blend (monsoon Pearson); other stations were 0.29-0.39 in Cell 4:")
print(pd.DataFrame(rows).to_string(index=False))

# --- 3b) year-by-year label check, before and after ---
def best_by_year(df, st="Sikles", min_n=60):
    refs = [x for x in refset if x != st]
    ref = df[refs].mean(axis=1, skipna=False)
    g = df[st]
    out = {}
    for y in range(1983, 2024):
        m = mon_mask & (yrs == y)
        if int((ref[m].notna() & g[m].notna()).sum()) < min_n:
            continue
        r = {"tm1": ref[m].corr(g.shift(1)[m]), "t": ref[m].corr(g[m]),
             "tp1": ref[m].corr(g.shift(-1)[m])}
        out[y] = max(r, key=r.get)
    return pd.Series(out)

before, after = best_by_year(rain_w), best_by_year(aligned)
print("\n3b) Years by best matching day (Sikles vs reference), Jun-Sep:")
print(pd.DataFrame({"original": before.value_counts(), "corrected": after.value_counts()}).fillna(0).astype(int).to_string())

# --- 4) remaining flagged years ---
print("\n4) Years still flagged best = tm1 after correction:", list(after[after == "tm1"].index))
print("   Years flagged best = tp1 after correction:", list(after[after == "tp1"].index))

Other six stations unchanged: True
Sikles: 3772 days in shifted ranges; extra NaN days created: 3
Saved: rain_wide_aligned.parquet, date_fixes.json

3a) Sikles vs CHIRPS blend (monsoon Pearson); other stations were 0.29-0.39 in Cell 4:
period (Jun-Sep days)  Sikles~CHIRPS orig  Sikles~CHIRPS corrected
     1983-01..2004-12               0.268                    0.268
2005-01..2011-05 (S1)               0.207                    0.362
     2011-06..2020-01               0.300                    0.300
2020-02..2023-12 (S2)               0.129                    0.232
       full 1983-2023               0.249                    0.283

3b) Years by best matching day (Sikles vs reference), Jun-Sep:
     original  corrected
t          29         38
tm1        11          2

4) Years still flagged best = tm1 after correction: [1997, 2002]
   Years flagged best = tp1 after correction: []


In [15]:
# ============================================================
# CELL 13 (02_satellite_extraction): Sikles flag check + ERA5-Land daily extraction
# What it does:
#   A) prints Sikles around 2009-08-27..09-01 (stored, corrected) next
#      to the other six stations: is the 99 mm now on a day when the
#      neighbors also had heavy rain?
#   B) loads ECMWF/ERA5_LAND/DAILY_AGGR, lists bands, requires the
#      core bands (precipitation sum, 2 m temperature, dewpoint, wind
#      u/v, surface pressure) and uses optional ones (daily min/max
#      temperature, soil water layer 1) when present
#   C) pulls daily values at the centre pixel of all 8 locations for
#      1983-2023 in 8-year chunks, one file per location in
#      data/satellite/era5land_daily/ (skips existing files; a failing
#      request is split in halves)
#   D) combines everything into data/satellite/era5land_daily_raw.parquet
#   E) comparison (monsoon Pearson, aligned gauges): ERA5-Land rain blend
#      vs gauge, CHIRPS blend vs gauge, ERA5-Land vs CHIRPS, and the
#      average of the two vs gauge; plus mean annual ERA5-Land rainfall
#      for all 8 locations
# Needs ee, pts, centers, test_stations, feat, rain_w, mon_mask, SAT_DIR.
# ============================================================
import time

aligned = pd.read_parquet(PROCESSED_DIR / "rain_wide_aligned.parquet")

# --- A) the 99 mm flag ---
w = slice("2009-08-27", "2009-09-01")
view = pd.concat([rain_w["Sikles"].loc[w].rename("Sikles stored"),
                  aligned["Sikles"].loc[w].rename("Sikles corrected"),
                  rain_w[[c for c in rain_w.columns if c != "Sikles"]].loc[w]], axis=1)
print("A) Sikles around the flagged day (mm):")
print(view.round(1).to_string())

# --- B) dataset and bands ---
ERA_D = "ECMWF/ERA5_LAND/DAILY_AGGR"
era_d = ee.ImageCollection(ERA_D)
first_d = era_d.filterDate("2021-07-07", "2021-07-08").first()
names_d = first_d.bandNames().getInfo()
for key in ("precipitation", "temperature_2m", "dewpoint", "wind", "pressure", "soil_water"):
    print(f"Bands containing '{key}':", [n for n in names_d if key in n])
print("Pixel size (m):", round(first_d.select(names_d[0]).projection().nominalScale().getInfo(), 0))

NEEDED = ["total_precipitation_sum", "temperature_2m", "dewpoint_temperature_2m",
          "u_component_of_wind_10m", "v_component_of_wind_10m", "surface_pressure"]
OPTIONAL = ["temperature_2m_max", "temperature_2m_min", "volumetric_soil_water_layer_1"]
missing = [b for b in NEEDED if b not in names_d]
if missing:
    raise ValueError(f"Required bands not found: {missing}. Pick the right names from the lists above.")
bands = NEEDED + [b for b in OPTIONAL if b in names_d]
print("Using bands:", bands)

# --- C) extraction ---
ERA_D_DIR = SAT_DIR / "era5land_daily"
ERA_D_DIR.mkdir(exist_ok=True)

def fetch_daily(geom, t0, t1):
    try:
        data = (era_d.select(bands).filterDate(ee.Date(t0), ee.Date(t1))
                     .getRegion(geom, 11132).getInfo())
    except Exception:
        span = pd.Timestamp(t1) - pd.Timestamp(t0)
        if span <= pd.Timedelta(days=200):
            raise
        mid = (pd.Timestamp(t0) + span / 2).floor("D").strftime("%Y-%m-%dT%H:%M:%S")
        return pd.concat([fetch_daily(geom, t0, mid), fetch_daily(geom, mid, t1)])
    d = pd.DataFrame(data[1:], columns=data[0])
    d["date"] = pd.to_datetime(d["time"], unit="ms").dt.normalize()
    return d.set_index("date")[bands].astype(float).sort_index()

chunks = [(1983, 1990), (1991, 1998), (1999, 2006), (2007, 2014), (2015, 2022), (2023, 2023)]
t_start = time.time()
for st in pts.index:
    path = ERA_D_DIR / f"{st.replace(' ', '_')}.parquet"
    if path.exists():
        print(f"{st}: already saved, skipped")
        continue
    geom = ee.Geometry.Point([float(pts.loc[st, "lon"]), float(pts.loc[st, "lat"])])
    parts = [fetch_daily(geom, f"{y0}-01-01T00:00:00", f"{y1 + 1}-01-01T00:00:00")
             for y0, y1 in chunks]
    d = pd.concat(parts).sort_index()
    d = d[~d.index.duplicated(keep="first")]
    d.to_parquet(path)
    print(f"{st}: {len(d)} days | elapsed {time.time() - t_start:.0f}s")

# --- D) combine ---
frames = []
for st in pts.index:
    d = pd.read_parquet(ERA_D_DIR / f"{st.replace(' ', '_')}.parquet")
    d.insert(0, "station", st)
    frames.append(d.reset_index())
era_raw = pd.concat(frames, ignore_index=True)
era_raw.to_parquet(SAT_DIR / "era5land_daily_raw.parquet")
print("\nCombined:", era_raw.shape, "|", era_raw["date"].min().date(), "->", era_raw["date"].max().date())
print("Missing values per column:")
print(era_raw[bands].isna().sum().to_string())

# --- E) comparison ---
def pear(x, y, m):
    ok = x[m].notna() & y[m].notna()
    return round(x[m][ok].corr(y[m][ok]), 3) if ok.sum() >= 30 else np.nan

rows = []
for st in test_stations:
    e = era_raw[era_raw["station"] == st].set_index("date")["total_precipitation_sum"].mul(1000)
    e = e.reindex(rain_w.index)
    eb = 0.875 * e.shift(1) + 0.125 * e
    cb = feat[feat["station"] == st].set_index("date")["chirps_blend_c"].reindex(rain_w.index)
    g = aligned[st]
    rows.append({
        "station": st,
        "ERA5L~gauge": pear(g, eb, mon_mask),
        "CHIRPS~gauge": pear(g, cb, mon_mask),
        "ERA5L~CHIRPS": pear(cb, eb, mon_mask),
        "mean(ERA5L,CHIRPS)~gauge": pear(g, (eb + cb) / 2, mon_mask),
    })
print("\nE1) Monsoon Pearson, aligned gauges (rain blend = 0.875*D-1 + 0.125*D):")
print(pd.DataFrame(rows).to_string(index=False))

rows = []
for st in pts.index:
    e = era_raw[era_raw["station"] == st].set_index("date")["total_precipitation_sum"].mul(1000)
    e = e.reindex(rain_w.index)
    r = {"location": st, "ERA5L_mm_yr": round(e.mean() * 365.25)}
    if st in aligned.columns:
        ok = aligned[st].notna()
        r["gauge_mm_yr"] = round(aligned[st][ok].mean() * 365.25)
        r["ERA5L_same_days"] = round(e[ok].mean() * 365.25)
        r["ratio"] = round(r["ERA5L_same_days"] / r["gauge_mm_yr"], 2)
    rows.append(r)
print("\nE2) Mean annual rainfall at the centre pixel (ERA5-Land):")
print(pd.DataFrame(rows).to_string(index=False))

A) Sikles around the flagged day (mm):
            Sikles stored  Sikles corrected  Pokhara Airport  Kunchha  Bandipur  Lumle  Damauli  Lamachaur
date                                                                                                      
2009-08-27           45.0               6.7              8.3     15.5       3.1   51.2     40.4       10.6
2009-08-28            1.6              45.0              1.8    154.3       0.2   60.5      0.0       16.2
2009-08-29           99.0               1.6              0.0      0.0       0.0    1.3      0.0        0.0
2009-08-30           28.7              99.0              0.0      0.0       0.0  174.0      0.0        0.0
2009-08-31            0.0              28.7             93.5      0.0     113.0  103.6     20.2      126.0
2009-09-01           11.6               0.0              0.0      0.0      43.0    0.5      0.0       84.0
Bands containing 'precipitation': ['total_precipitation_sum', 'total_precipitation_min', 'total_precipita

In [16]:
# ============================================================
# CELL 14 (02_satellite_extraction): Daily-aggregate mapping and cost check (2006-2012)
# What it does:
#   1) loads from disk: rain_wide_aligned.parquet (gauges), era5land_daily_raw.parquet
#      (ERA5-Land daily aggregate), chirps_features.parquet, and the hourly
#      ERA5-Land precipitation files for 2006-2012 downloaded in Cell 7
#   2) mapping check: compares the daily-aggregate value for date D with
#      hourly 24 h sums ending at D+1 00:00 UTC (stamps D 01:00..D+1 00:00)
#      and ending at D 23:00 UTC (stamps D 00:00..D 23:00); the one with
#      correlation ~1 and tiny differences is the real definition
#   3) cost check, monsoon Pearson against the aligned gauges over 2006-2012:
#        daily blend     = 0.875*daily(D-1) + 0.125*daily(D)    (what E1 used)
#        hourly 03Z      = exact 24 h window ending D 03:00 UTC
#        CHIRPS blend    = chirps_blend_c
#        avg(hourly03Z, CHIRPS) and avg(daily blend, CHIRPS)
#   no Earth Engine needed
# ============================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)

ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"
SAT_DIR = ROOT / "data" / "satellite"
ERA_DIR = SAT_DIR / "era5land_hourly_precip"

aligned = pd.read_parquet(PROCESSED_DIR / "rain_wide_aligned.parquet")
era_raw = pd.read_parquet(SAT_DIR / "era5land_daily_raw.parquet")
feat = pd.read_parquet(SAT_DIR / "chirps_features.parquet")
stations7 = list(aligned.columns)

days = pd.date_range("2006-01-01", "2012-12-31")
mon = pd.Series(days.month.isin([6, 7, 8, 9]), index=days)

def load_hourly(st):
    parts = [pd.read_parquet(ERA_DIR / f"{st.replace(' ', '_')}_{y}.parquet")["mm"]
             for y in range(2006, 2013)]
    h = pd.concat(parts).sort_index()
    h = h[~h.index.duplicated(keep="first")]
    return h.reindex(pd.date_range(h.index.min(), h.index.max(), freq="h"))

def pear(x, y, m):
    ok = m & x.notna() & y.notna()
    return round(x[ok].corr(y[ok]), 3) if ok.sum() >= 30 else np.nan

map_rows, cost_rows = [], []
for st in stations7:
    s24 = load_hourly(st).rolling(24, min_periods=24).sum()
    daily = (era_raw[era_raw["station"] == st].set_index("date")["total_precipitation_sum"]
             .mul(1000).reindex(pd.date_range("2005-12-31", "2012-12-31")))
    d = daily.reindex(days)
    d_blend = (0.875 * daily.shift(1) + 0.125 * daily).reindex(days)

    w24 = pd.Series(s24.reindex(days + pd.Timedelta(hours=24)).values, index=days)   # stamps D 01:00..D+1 00:00
    w23 = pd.Series(s24.reindex(days + pd.Timedelta(hours=23)).values, index=days)   # stamps D 00:00..D 23:00
    w03 = pd.Series(s24.reindex(days + pd.Timedelta(hours=3)).values, index=days)    # window ending D 03:00 UTC

    allm = pd.Series(True, index=days)
    map_rows.append({
        "station": st,
        "r(daily, end D+1 00Z)": pear(d, w24, allm), "mean|diff| mm": round((d - w24).abs().mean(), 3),
        "r(daily, end D 23Z)": pear(d, w23, allm), "mean|diff| mm ": round((d - w23).abs().mean(), 3),
    })

    g = aligned[st].reindex(days)
    c = feat[feat["station"] == st].set_index("date")["chirps_blend_c"].reindex(days)
    cost_rows.append({
        "station": st,
        "daily blend": pear(g, d_blend, mon),
        "hourly 03Z": pear(g, w03, mon),
        "CHIRPS blend": pear(g, c, mon),
        "avg(hourly03Z, CHIRPS)": pear(g, (w03 + c) / 2, mon),
        "avg(daily blend, CHIRPS)": pear(g, (d_blend + c) / 2, mon),
    })

print("2) Which hourly window equals the daily-aggregate value for date D? (2006-2012, all days)")
print(pd.DataFrame(map_rows).to_string(index=False))
print("\n3) Monsoon Pearson vs aligned gauges, 2006-2012 (same days for every column):")
print(pd.DataFrame(cost_rows).to_string(index=False))

2) Which hourly window equals the daily-aggregate value for date D? (2006-2012, all days)
        station  r(daily, end D+1 00Z)  mean|diff| mm  r(daily, end D 23Z)  mean|diff| mm 
Pokhara Airport                  0.996          0.804                  1.0             0.0
        Kunchha                  0.996          0.639                  1.0             0.0
       Bandipur                  0.995          0.464                  1.0             0.0
          Lumle                  0.995          0.587                  1.0             0.0
        Damauli                  0.995          0.497                  1.0             0.0
         Sikles                  0.996          0.503                  1.0             0.0
      Lamachaur                  0.996          0.690                  1.0             0.0

3) Monsoon Pearson vs aligned gauges, 2006-2012 (same days for every column):
        station  daily blend  hourly 03Z  CHIRPS blend  avg(hourly03Z, CHIRPS)  avg(daily blend, CHIRPS

In [17]:
# ============================================================
# CELL 15 (02_satellite_extraction): Skill by period + ERA5-Land features + merged satellite table
# What it does (reads everything from disk; no Earth Engine needed):
#   1) loads aligned gauges, ERA5-Land daily raw table and CHIRPS features
#   2) skill by period: monsoon Pearson of each gauge vs CHIRPS blend and vs
#      ERA5-Land rain blend for 1983-92, 1993-2002, 2003-12, 2013-23
#   3) builds ERA5-Land features per location. All variables use the blend
#      0.833*(D-1) + 0.167*(D), matching the gauge window D-1 04:00..D 03:00 UTC:
#        era_p_blend (mm), era_p_dm1, era_p_d0, era_p_blend_prev, era_p_blend_sum3
#        era_t2m_c, era_dtr_c (daily max-min), era_dewdep_c (T - dewpoint),
#        era_rh_pct, era_u10, era_v10, era_wspd, era_sp_hpa, era_dsp_hpa
#        (pressure change vs the previous blended day), era_soil_w1
#   4) merges with chirps_features on (station, date) and saves
#      data/satellite/satellite_features.parquet
#   5) sanity prints: shape, missing values, and mean temperature,
#      humidity and surface pressure per location
# ============================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)

ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"
SAT_DIR = ROOT / "data" / "satellite"

aligned = pd.read_parquet(PROCESSED_DIR / "rain_wide_aligned.parquet")
era_raw = pd.read_parquet(SAT_DIR / "era5land_daily_raw.parquet")
chirps_f = pd.read_parquet(SAT_DIR / "chirps_features.parquet")

W_PREV, W_CUR = 20 / 24, 4 / 24

def blend(x):
    return W_PREV * x.shift(1) + W_CUR * x

# --- 3) ERA5-Land features ---
frames = []
for st, d in era_raw.groupby("station"):
    d = d.set_index("date").sort_index()
    p = d["total_precipitation_sum"] * 1000                      # m -> mm
    T = d["temperature_2m"] - 273.15
    Td = d["dewpoint_temperature_2m"] - 273.15
    rh = 100 * np.exp(17.625 * Td / (243.04 + Td)) / np.exp(17.625 * T / (243.04 + T))
    u, v = d["u_component_of_wind_10m"], d["v_component_of_wind_10m"]
    sp = d["surface_pressure"] / 100                             # Pa -> hPa
    f = pd.DataFrame({
        "era_p_blend": blend(p),
        "era_p_dm1": p.shift(1),
        "era_p_d0": p,
        "era_t2m_c": blend(T),
        "era_dtr_c": blend(d["temperature_2m_max"] - d["temperature_2m_min"]),
        "era_dewdep_c": blend(T - Td),
        "era_rh_pct": blend(rh),
        "era_u10": blend(u),
        "era_v10": blend(v),
        "era_wspd": blend(np.sqrt(u ** 2 + v ** 2)),
        "era_sp_hpa": blend(sp),
        "era_soil_w1": blend(d["volumetric_soil_water_layer_1"]),
    })
    f["era_p_blend_prev"] = f["era_p_blend"].shift(1)
    f["era_p_blend_sum3"] = f["era_p_blend"].rolling(3, min_periods=3).sum()
    f["era_dsp_hpa"] = f["era_sp_hpa"] - f["era_sp_hpa"].shift(1)
    f.insert(0, "station", st)
    f.index.name = "date"
    frames.append(f.reset_index())
era_f = pd.concat(frames, ignore_index=True)

# --- 2) skill by period ---
blocks = [("1983-1992", 1983, 1992), ("1993-2002", 1993, 2002),
          ("2003-2012", 2003, 2012), ("2013-2023", 2013, 2023)]
mon = pd.Series(aligned.index.month.isin([6, 7, 8, 9]), index=aligned.index)
yrs = pd.Series(aligned.index.year, index=aligned.index)
stations7 = list(aligned.columns)

def pear(x, y, m):
    ok = m & x.notna() & y.notna()
    return round(x[ok].corr(y[ok]), 3) if ok.sum() >= 30 else np.nan

rows_c, rows_e = [], []
for lab, y0, y1 in blocks:
    m = mon & (yrs >= y0) & (yrs <= y1)
    rc, re = {"block": lab}, {"block": lab}
    for st in stations7:
        g = aligned[st]
        c = chirps_f[chirps_f["station"] == st].set_index("date")["chirps_blend_c"].reindex(aligned.index)
        e = era_f[era_f["station"] == st].set_index("date")["era_p_blend"].reindex(aligned.index)
        rc[st], re[st] = pear(g, c, m), pear(g, e, m)
    rc["mean"] = round(np.nanmean([rc[s] for s in stations7]), 3)
    re["mean"] = round(np.nanmean([re[s] for s in stations7]), 3)
    rows_c.append(rc)
    rows_e.append(re)
print("2a) Monsoon Pearson: gauge vs CHIRPS blend, by period")
print(pd.DataFrame(rows_c).to_string(index=False))
print("\n2b) Monsoon Pearson: gauge vs ERA5-Land rain blend (20/24, 4/24), by period")
print(pd.DataFrame(rows_e).to_string(index=False))

# --- 4) merge and save ---
sat = chirps_f.merge(era_f, on=["station", "date"], how="outer").sort_values(["station", "date"])
sat = sat.reset_index(drop=True)
sat.to_parquet(SAT_DIR / "satellite_features.parquet")
print("\nsatellite_features:", sat.shape, "|", sat["date"].min().date(), "->", sat["date"].max().date())
print("Locations:", sat["station"].nunique(), "| duplicate (station, date):",
      int(sat.duplicated(["station", "date"]).sum()))
print("\nMissing values per column:")
print(sat.drop(columns=["station", "date"]).isna().sum().to_string())

# --- 5) sanity ---
summ = sat.groupby("station")[["era_t2m_c", "era_rh_pct", "era_sp_hpa", "era_wspd"]].mean().round(1)
print("\nMean temperature (C), relative humidity (%), surface pressure (hPa) and wind speed (m/s):")
print(summ.to_string())
print("\nSaved: data/satellite/satellite_features.parquet")

2a) Monsoon Pearson: gauge vs CHIRPS blend, by period
    block  Pokhara Airport  Kunchha  Bandipur  Lumle  Damauli  Sikles  Lamachaur  mean
1983-1992            0.378    0.346     0.223  0.292    0.375   0.262      0.210 0.298
1993-2002            0.415    0.329     0.340  0.393    0.360   0.280      0.291 0.344
2003-2012            0.403    0.312     0.307  0.305    0.452   0.298      0.360 0.348
2013-2023            0.365    0.296     0.411  0.345    0.392   0.294      0.297 0.343

2b) Monsoon Pearson: gauge vs ERA5-Land rain blend (20/24, 4/24), by period
    block  Pokhara Airport  Kunchha  Bandipur  Lumle  Damauli  Sikles  Lamachaur  mean
1983-1992            0.331    0.209     0.161  0.257    0.268   0.268      0.238 0.247
1993-2002            0.409    0.254     0.217  0.325    0.323   0.274      0.258 0.294
2003-2012            0.404    0.272     0.302  0.264    0.361   0.306      0.324 0.319
2013-2023            0.433    0.263     0.405  0.362    0.327   0.335      0.341 0.352